# Name That LLM — first baseline

CPU, internet disabled, competition data only. This notebook trains on all labeled passages and predicts every runtime test row.

Chosen variant: `char`. Local five-fold prompt-grouped balanced accuracy: **31.0410%**. This is a development estimate, not a leaderboard score. Related topics can cross prompt groups; the full dataset informed the earlier qualitative review.

The three visible test passages check the file format only. Kaggle supplies the hidden test set during scoring. Output: `submission.parquet`.

In [ ]:
"""Shared training and submission code, also embedded in the Kaggle notebook."""

from __future__ import annotations

import hashlib
import json
import logging
import platform
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

LOGGER = logging.getLogger("name_that_llm")
LABELS = tuple(sorted((
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",
    "gpt-6-astra", "gemini-3.1-pro-preview", "gemini-3.6-flash",
    "gemini-3.7-flash", "gemini-3.8-flash", "claude-fable-5",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",
    "claude-sonnet-5", "qwen3.8-max", "glm-5.3",
    "glm-5.3-flash", "kimi-k3",
)))


@dataclass(frozen=True)
class BaselineConfig:
    """Fixed before inspecting any validation results."""

    variant: str = "char"
    seed: int = 42
    n_splits: int = 5
    c: float = 1.0
    min_df: int = 2
    char_max_features: int = 250_000
    word_max_features: int = 100_000
    max_iter: int = 10_000
    tolerance: float = 1e-4


def validate_frame(frame: pd.DataFrame, *, training: bool) -> None:
    required = ["id", "text"] + (["prompt_id", "model"] if training else [])
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("Input must contain at least one passage.")
    for column in required:
        if frame[column].isna().any() or not frame[column].map(
            lambda value: isinstance(value, str) and bool(value.strip())
        ).all():
            raise ValueError(f"Column {column!r} contains missing, empty or non-string values.")
    if frame["id"].duplicated().any():
        raise ValueError("Passage IDs must be unique.")
    if training and set(frame["model"]) != set(LABELS):
        raise ValueError("Training labels must match all 18 competition labels.")


def load_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Explicit string types retain leading zeros and the exact CSV-decoded text.
    train = pd.read_csv(data_dir / "train.csv", dtype=str, keep_default_na=False)
    test = pd.read_csv(data_dir / "test.csv", dtype=str, keep_default_na=False)
    validate_frame(train, training=True)
    validate_frame(test, training=False)
    return train, test


def build_pipeline(config: BaselineConfig) -> Pipeline:
    if config.variant not in {"char", "char_word"}:
        raise ValueError(f"Unknown baseline variant: {config.variant}")
    common = dict(min_df=config.min_df, sublinear_tf=True, dtype=np.float32)
    features = [("char", TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), lowercase=False,
        strip_accents=None, max_features=config.char_max_features, **common,
    ))]
    if config.variant == "char_word":
        features.append(("word", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), lowercase=True,
            strip_accents=None, stop_words=None,
            max_features=config.word_max_features, **common,
        )))
    return Pipeline([
        ("features", FeatureUnion(features)),
        ("normalize", Normalizer()),
        ("classifier", LinearSVC(
            C=config.c, dual=True, random_state=config.seed,
            max_iter=config.max_iter, tol=config.tolerance,
        )),
    ])


def make_folds(train: pd.DataFrame, config: BaselineConfig) -> list[tuple[np.ndarray, np.ndarray]]:
    validate_frame(train, training=True)
    splitter = GroupKFold(
        n_splits=config.n_splits, shuffle=True, random_state=config.seed,
    )
    folds = list(splitter.split(train["text"], train["model"], groups=train["prompt_id"]))
    counts = np.zeros(len(train), dtype=int)
    for fitting, held_out in folds:
        if set(train.iloc[fitting]["prompt_id"]) & set(train.iloc[held_out]["prompt_id"]):
            raise ValueError("A prompt appears in both sides of a fold.")
        for indexes in (fitting, held_out):
            if set(train.iloc[indexes]["model"]) != set(LABELS):
                raise ValueError("Every fold must contain all 18 labels on both sides.")
        counts[held_out] += 1
    if not np.all(counts == 1):
        raise ValueError("Each passage must be held out exactly once.")
    return folds


def fit_checked(pipeline: Pipeline, text: pd.Series, labels: pd.Series) -> Pipeline:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pipeline.fit(text, labels)
    for warning in caught:
        LOGGER.warning("%s: %s", warning.category.__name__, warning.message)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError("LinearSVC did not converge; inspect the run log before proceeding.")
    return pipeline


def validate_submission(
    test: pd.DataFrame, submission: pd.DataFrame,
) -> None:
    validate_frame(test, training=False)
    if list(submission.columns) != ["id", "model"]:
        raise ValueError("Submission columns must be exactly id, model.")
    if len(submission) != len(test):
        raise ValueError("Submission row count differs from the runtime test set.")
    if submission.isna().any().any() or submission["id"].duplicated().any():
        raise ValueError("Submission contains missing values or duplicate IDs.")
    if submission["id"].tolist() != test["id"].tolist():
        raise ValueError("Submission IDs must match runtime test IDs in order.")
    if not set(submission["model"]).issubset(LABELS):
        raise ValueError("Submission contains an unknown model label.")


def predict_submission(pipeline: Pipeline, test: pd.DataFrame) -> pd.DataFrame:
    validate_frame(test, training=False)
    predictions = pipeline.predict(test["text"])
    submission = pd.DataFrame({"id": test["id"].to_numpy(), "model": predictions})
    validate_submission(test, submission)
    return submission


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def runtime_versions() -> dict[str, str]:
    return {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    }



In [ ]:
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
config = BaselineConfig(**json.loads('{"c": 1.0, "char_max_features": 250000, "max_iter": 10000, "min_df": 2, "n_splits": 5, "seed": 42, "tolerance": 0.0001, "variant": "char", "word_max_features": 100000}'))
override = os.environ.get("NAME_THAT_LLM_DATA_DIR")
candidates = [Path(override)] if override else [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
]
data_dir = next((path for path in candidates
                 if (path / "train.csv").is_file() and (path / "test.csv").is_file()), None)
if data_dir is None:
    raise FileNotFoundError("Attach the name-that-llm competition data.")
train, test = load_data(data_dir)
print(json.dumps({"versions": runtime_versions(), "config": asdict(config),
                  "train_rows": len(train), "test_rows": len(test)}, indent=2))
model = fit_checked(build_pipeline(config), train["text"], train["model"])
submission = predict_submission(model, test)
submission.to_parquet("submission.parquet", index=False)
validate_submission(test, pd.read_parquet("submission.parquet"))
print(f"Verified submission.parquet: {len(submission)} rows")
